# DFT diagnostic MCMC (refloxide)

Loads pickled DFT and free-tensor fits, builds a **single-energy 283.7 eV** objective with the Rust refloxide reflectivity kernel, and runs emcee for structure uncertainties.

The old multi-energy `GlobalObjective` path duplicated parameters (31 walkers dimensions) and evaluated stock pyref reflectivity on every energy each step (~27 s/emcee iteration). This notebook uses `build_dft_diagnostic_pyref_mcmc_objective` (~5 ms/logl, 13 DOF).

For multi-energy **refits**, use `fit_dft_refloxide.ipynb` (`BatchedGlobalObjective` + shared `DispersiveStructure`).


In [ ]:
import time

import matplotlib.pyplot as plt

from utils import (
    build_dft_diagnostic_pyref_mcmc_objective,
    read_fit,
    read_ooc,
)
from utils.helpers.plotting_helper import set_plotting_defaults
from utils.models import CurveFitter, configure_refloxide_fitting

configure_refloxide_fitting(use_rust=True, parallel=False)

plt.style.use(["science", "no-latex"])
set_plotting_defaults()

In [ ]:
dft_constrained = read_fit("dft/dft_en_offset_new2.pkl", material="znpc")
free = read_fit("free/free_en_offset_init_2.pkl", material="znpc")
oocs = read_ooc("dft.csv", material="znpc")

In [ ]:
mcmc_obj = build_dft_diagnostic_pyref_mcmc_objective(
    dft_constrained,
    oocs,
    free_bundle=free,
)
mcmc_obj.logl()


In [ ]:
mcmc_obj.varying_parameters()

In [ ]:
print(len(mcmc_obj.varying_parameters()))

t0 = time.perf_counter()
for _ in range(20):
    mcmc_obj.logl()
print(f"logl: {(time.perf_counter() - t0) / 20 * 1e3:.1f} ms")

In [ ]:
quick_fit = CurveFitter(mcmc_obj, walkers_per_param=10)

In [ ]:
quick_fit.initialise(pos="jitter", random_state=0)
quick_fit.sample(100, pool=1, skip_check=True)

In [ ]:
import matplotlib.pyplot as plt

fig = plt.plot(quick_fit.logpost)

In [ ]:
obj = quick_fit.objective
obj.plot()